[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/compute_module1/blob/main/notebooks/lab_04_the_notebook.ipynb)

# Lab 4: The Notebook

*From* **A First Taste of Algorithms — and the Machines That Learn** *(The Curious Family Library). A computational lab: run the cells in order. A few labs end with a deliberately broken “Break it” cell — that one is meant to fail, and seeing how is the point.*

*Accompanies Chapter 4. The chapter caught a correct recursion wasting its life on repetition, and tore the exponential down with a notebook. In this lab the waste is counted call by call, the notebook is built, and your own Chapter 2 greedy is put on trial against a changer that cannot lose.*

## 1. The amnesiac, counted

The staircase recursion, plus the counting trick from Lab 3 — each call reports the *total number of calls* it caused, alongside its answer, as a two-item list `[answer, calls]`:

In [ ]:
def ways_count(n):
    if n <= 2:
        return [n, 1]                     # ways(1)=1, ways(2)=2; one call
    a = ways_count(n - 1)
    b = ways_count(n - 2)
    return [a[0] + b[0], a[1] + b[1] + 1]

print(ways_count(6))
print(ways_count(10))

Check ways(6) against the chapter's hand count — and check the call count against your drawing of the tree (the chapter said fifteen boxes; does the machine agree?). Now the audit, in the house style:

In [ ]:
for n in [10, 15, 20, 25, 30]:
    result = ways_count(n)
    print(n, "ways:", result[0], "  calls:", result[1])

Read the calls column the way you read every table in this book: n goes up by 5 each row — what happens to the calls? This is not Chapter 1's logarithm and not even Chapter 3's doubling-per-step; measure its gait (each row is roughly *how many times* the previous?). Run n = 34 and *feel* the pause before the answer — the first time in this book the machine has kept you waiting. The answer is a seven-digit number you could extend by hand in seconds. All that waiting was repetition.

## 2. The notebook

Python's notebook is the **dictionary**: like a list, but indexed by *question* rather than by position. Three moves to learn, all visible here:

In [ ]:
memo = {}                 # an empty notebook
memo[7] = 13              # write: next to question 7, answer 13
print(7 in memo)          # ask: is question 7 already answered?
print(memo[7])            # read the answer

Now the chapter's fix, verbatim — look it up, or compute-and-write:

In [ ]:
memo = {1: 1, 2: 2}       # the floor, pre-written

def ways_memo(n):
    if n in memo:
        return memo[n]
    answer = ways_memo(n - 1) + ways_memo(n - 2)
    memo[n] = answer
    return answer

print(ways_memo(30))

Same recursion, same floor, same answers — verify a few against §1. The only new behavior: no question is ever *computed* twice.

## 3. The collapse, measured

Count the memoized version's calls (the counter rides along in a one-item list — a small trick: numbers passed to functions don't come back changed, but a list's *contents* do, so we keep the count in a box and pass the box):

In [ ]:
memo = {1: 1, 2: 2}
calls = [0]

def ways_memo_count(n):
    calls[0] = calls[0] + 1
    if n in memo:
        return memo[n]
    answer = ways_memo_count(n - 1) + ways_memo_count(n - 2)
    memo[n] = answer
    return answer

ways_memo_count(30)
print("memoized calls for n=30:", calls[0])

Set it beside §1's number for n = 30. Millions against dozens, identical answers. Now the gait check: reset the notebook (rerun the cell after changing 30 to 40, but reset `memo` and `calls` first — why must you reset? predict what the count reports if you *don't*) and confirm the memoized call count grows like n itself: the straight line that replaced the explosion.

## 4. Sevenia on trial

The chapter's recurrence, as a program — for *any* coin system. The notebook here must be reset per problem, so we wrap the whole trial in one function that builds a fresh memo, then defines the recursive worker inside (a function defined inside a function: new sight, no new idea — the inner worker simply gets to use the outer function's `memo` and `coins`):

In [ ]:
def fewest_coins(coins, amount):
    memo = {0: 0}                       # the floor: no debt, no coins

    def best(a):
        if a in memo:
            return memo[a]
        options = []
        for c in coins:
            if c <= a:
                options.append(best(a - c))
        answer = 1 + min(options)       # min: the cheapest priced future
        memo[a] = answer
        return answer

    return best(amount)

print(fewest_coins([10, 7, 1], 14))

Two. The wound closes on your screen. (`min` of a list is what it says; it has been waiting in Python all along for this chapter.)

Now the full trial — your Chapter 2 greedy versus the table method, every amount to 100. Retype your greedy from Lab 2 (or paste it from your notebook there):

In [ ]:
def greedy_change(coins, amount):
    used = []
    for coin in coins:
        while amount >= coin:
            amount = amount - coin
            used.append(coin)
    return used

for amount in range(1, 101):
    g = len(greedy_change([10, 7, 1], amount))
    d = fewest_coins([10, 7, 1], amount)
    if g != d:
        print(amount, "greedy:", g, "  table:", d)

Compare the verdict list with your Lab 2 audit (which used brute force as the truth). Same failures, same truth — but notice what changed *about the truth-machine*: Lab 2's three nested loops tried everything; this one prices each amount once. The notebook replaced the brute force. Confirm the new machine's reach: `fewest_coins([10, 7, 1], 900)` — instant, where three nested loops would grind.

## 5. The same table, bottom-up

The chapter promised the notebook can be filled by a plain loop, floor upward, no recursion at all:

In [ ]:
def fewest_coins_table(coins, amount):
    best = [0]                          # best[0] = 0; the table grows rightward
    for a in range(1, amount + 1):
        options = []
        for c in coins:
            if c <= a:
                options.append(best[a - c])
        best.append(1 + min(options))
    return best

table = fewest_coins_table([10, 7, 1], 14)
print(table)
print("best(14) =", table[14])

Read the printed list against the chapter's hand-built table — entry by entry, they must agree (the table *is* the method; the two routes merely fill it in different orders). This version climbs to any height without a call stack: try amount 5,000, then try the same in §4's recursive version and witness which one meets Lab 3's old enemy.

## Break it

Predict first. Run second. Explain any gap third.

**1.** In §2's `ways_memo`, delete only the line `memo[n] = answer` (keep everything else). Predict: still correct? still fast? one or the other? Run §3's count with the line gone. Which single line of this lab was the entire chapter?

**2.** Sabotage by sharing: move §4's `memo = {0: 0}` *outside* the function (one notebook for all trials), then run `fewest_coins([10, 7, 1], 14)` followed by `fewest_coins([25, 10, 5, 1], 14)`. Predict both outputs first. The second answer is wrong — wrong *how*, and in whose favor? State the moral as one sentence about what a notebook entry silently assumes. (This bug, in grown-up systems, is called cache poisoning, and it has caused real outages; you have now caused one in eleven lines.)

**3.** Hops of {2, 3} only: adapt `ways_memo` for a staircase where you may hop 2 or 3 steps. The floor needs care — decide what ways(0) and ways(1) should *be* (is "do nothing" a way to climb zero steps? is 1 climbable at all?), write your floor, and test against hand counts for n = 2, 3, 4, 5. Then break a classmate's floor: find the n where a plausible-but-wrong floor (say, ways(0) = 0) first gives a wrong count. Floors are where recursions are won and lost.

**4.** In §5's bottom-up changer, feed it a kingdom without the 1-coin: `fewest_coins_table([10, 7], 11)`. Predict the failure mode precisely — wrong answer, error, or silence? — and read what actually happens to its last line. Then repair the design: unpayable amounts need an answer that survives `min` and `+ 1` honestly (the chapter's Problem 3.5 is circling this same hole). Propose a fix, implement it, and verify amount 11 reports "impossible" while 17 reports 2.



<div style="page-break-after: always;"></div>